## Prediction File for ANN Model
---

In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model

import pandas as pd 
import numpy as np
import pickle

> Loading Pickle file
---

In [2]:
# Trained Model
model = load_model('models/ann-customer-churn-prediction-model.h5')

# Encoder
with open('models/ann-customer-churn-prediction-onehot_encoder_geo.pkl', 'rb') as file:
    onehot_encoder_geo = pickle.load(file)

# Gender
with open('models/ann-customer-churn-prediction-label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

# Scaler
with open('models/ann-customer-churn-prediction-scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)


> New Input Data
--- 

In [3]:
input_data = {
    'CreditScore':600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure':3,
    'Balance':60000,
    'NumOfProducts':2,
    'HasCrCard':1,
    'IsActiveMember':1,
    'EstimatedSalary':50000
}

input_df= pd.DataFrame([input_data])


> Encoding Categorical Variable: Gender
---

In [4]:
input_df['Gender'] = label_encoder_gender.transform([input_data['Gender']])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


> OHE: Geography
---


In [5]:
geo_encoded = onehot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

d:\CODE\ann-customer-churn-prediction\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


> Concatinating Geography Variable into DataFrame
---

In [6]:
input_df = pd.concat([input_df.drop("Geography", axis=1), geo_encoded_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


> Scaling Input Dataframe
---

In [7]:
input_scaled_df = scaler.transform(input_df)
input_scaled_df

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

> Predict Churn
---

In [8]:
prediction = model.predict(input_scaled_df)
print(prediction)

prediction_prob = prediction[0][0]
print(prediction_prob)

1/1 [==============================] - 0s 96ms/step
[[0.04288146]]
0.042881455


>Final Result
---

In [9]:
if prediction_prob > 0.5:
    print('Customer is likely to churn.')
else:
    print('Customer is not likely to churn.')

Customer is not likely to churn.
